### Fine-tuning: Roman-Urdu Sentiment Classifier

Continues fine-tuning `Khubaib01/roman-urdu-sentiment-xlm-r` on our own cleaned dataset,

#### 1. Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

#### 2. Confirm GPU is active
Should print something like `Tesla T4`. If it prints an error, go back and set the runtime type.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only -- go set Runtime > T4 GPU")

#### 3. Upload the cleaned dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # select roman_urdu_clean.csv

#### 4. Load & split data (SAME split as the rest of the project)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("roman_urdu_clean.csv")
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(y_train.value_counts())

#### 5. Load tokenizer + model
We start from the already-fine-tuned checkpoint and continue training it on our data,
rather than starting from a generic base model.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "Khubaib01/roman-urdu-sentiment-xlm-r"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# confirm the label mapping the model already uses
print(model.config.id2label)

#### 6. Tokenize the data
Converts text into the numeric input format the transformer expects (token IDs).
`padding=True` makes all sequences in a batch the same length;
`truncation=True` cuts off anything longer than the model's max length (safety net).

In [ ]:
from datasets import Dataset

# map our string labels to the model's existing label ids, so training
# targets match what the model already understands
label2id = {v: k for k, v in model.config.id2label.items()}
print("label2id:", label2id)

train_df = pd.DataFrame({"text": X_train, "label": y_train.map(label2id)})
test_df = pd.DataFrame({"text": X_test, "label": y_test.map(label2id)})

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

#### 7. Define evaluation metric

In [ ]:
import numpy as np
import evaluate

f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return f1_metric.compute(predictions=preds, references=labels, average="macro")

#### 8. Set training arguments
- `num_train_epochs=3` -- 3 full passes over the training data (a common starting point; too many
  epochs risks *overfitting*, where the model starts memorizing training examples instead of
  learning general patterns)
- `learning_rate=2e-5` -- a small, standard learning rate for fine-tuning transformers
  (fine-tuning uses a much smaller rate than training from scratch, since the model already
  has good weights we don't want to disrupt too aggressively)
- `fp16=True` -- uses 16-bit precision instead of 32-bit, roughly halving memory use and
  speeding up training on the T4 GPU, with minimal accuracy impact

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./finetuned_model",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

#### 9. Train
This is the actual fine-tuning step -- expect roughly 15-40 minutes on a free T4 GPU
for this dataset size. Watch the eval F1 printed after each epoch.

In [ ]:
trainer.train()

#### 10. Final evaluation on test set

In [ ]:
from sklearn.metrics import classification_report

preds_output = trainer.predict(test_ds)
preds = np.argmax(preds_output.predictions, axis=-1)

id2label = model.config.id2label
pred_labels = [id2label[p] for p in preds]
true_labels = [id2label[l] for l in test_ds["label"]]

print(classification_report(true_labels, pred_labels))

print("\nCompare against:")
print("  TF-IDF + Logistic Regression:              0.6349")
print("  Multilingual MiniLM embeddings + LR:        0.5272")
print("  Off-the-shelf pretrained transformer:       0.6662")

#### 11. Save and download the fine-tuned model

In [ ]:
trainer.save_model("./finetuned_model_final")
tokenizer.save_pretrained("./finetuned_model_final")

!zip -r finetuned_model_final.zip finetuned_model_final

from google.colab import files
files.download("finetuned_model_final.zip")